In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.metrics import (homogeneity_score,v_measure_score,adjusted_mutual_info_score,normalized_mutual_info_score,adjusted_rand_score,fowlkes_mallows_score)


In [ ]:
repeat_times = 1
simple_path = '/home/cavin/jt/benchmark/data/hbc/s1_filtered_cells.h5ad'
celltype_col = 'cell_type'
cell_emb_col = 'X_pca'
pca_components = 50
random_seed=2026 + repeat_times * 200
save_path = "/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_with_annotations/PCA_human_lung_cancer_emb.parquet"


In [ ]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

In [ ]:
adata.obsm[cell_emb_col] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

In [ ]:
res_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
key_added='leiden'
sc.pp.neighbors(adata, use_rep=cell_emb_col,random_state=random_seed)
adata


In [ ]:
max_value = 0
metrics = {"method": "PCA", "ARI": 0, "NMI": 0, "AMI": 0, "HS": 0, "VM": 0, "FMI": 0, "mean value": 0}
save_label_df = None

In [ ]:

for used_res in res_list:
    sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=key_added)
    true_labels = np.array(adata.obs[celltype_col])
    cluster_labels = np.array(adata.obs[key_added])
    FMI = fowlkes_mallows_score(true_labels, cluster_labels)
    homogeneity = homogeneity_score(true_labels, cluster_labels)
    v_measure = v_measure_score(true_labels, cluster_labels)
    ami = adjusted_mutual_info_score(true_labels, cluster_labels)
    nmi = normalized_mutual_info_score(true_labels, cluster_labels)
    ari = adjusted_rand_score(true_labels, cluster_labels)
    mean_value = np.mean([FMI, homogeneity, v_measure, ami, nmi, ari])

    n_cluster = len(adata.obs[key_added].unique())
    n_celltype = len(adata.obs[celltype_col].unique())
    if mean_value > max_value:
        save_label_df = adata.obs[key_added]
        metrics["ARI"] = ari
        metrics["NMI"] = nmi
        metrics["AMI"] = ami
        metrics["HS"] = homogeneity
        metrics["VM"] = v_measure
        metrics["FMI"] = FMI
        metrics["mean value"] = mean_value
        max_value = mean_value

    print(f'resolution = {used_res} | n_celltype = {n_celltype} | n_cluster = {n_cluster} | mean_value = {round(mean_value,3)} | ARI = {round(ari,3)} |  NMI = {round(nmi,3)} | AMI = {round(ami,3)} | HS = {round(homogeneity,3)} | VM = {round(v_measure,3)} | FMI = {round(FMI,3)} ')

In [ ]:
metrics

In [ ]:
save_label_df

In [ ]:
save_label_df_path = f"/home/cavin/jt/benchmark/experiments/results/labels_df/breast_cancer/pca_human_lung_cancer_labels_repeat_{repeat_times}.csv"

In [ ]:
save_label_df.to_csv(save_label_df_path)

In [ ]:
metrics_df = pd.DataFrame.from_dict(metrics, orient='index').T

In [ ]:
metrics_df_save_path = f"/home/cavin/jt/benchmark/experiments/results/cluster_metrics/breast_cancer/human_breast_cancer_metrics_repeat_{repeat_times}.csv"

In [ ]:
metrics_df.to_csv(metrics_df_save_path, index=False,mode="a", header=not pd.io.common.file_exists(metrics_df_save_path))